# Trace-to-Specialist V1 — Kaggle (free T4)

Vertical slice: financial metric extraction → Base vs SFT vs GRPO on frozen test.

**Before run:** Settings → Accelerator = GPU T4. Add secret `HF_TOKEN` (Settings → Add-ons → Secrets) — Qwen checkpoints are gated.

**Repo:** either `!git clone <your-repo-url>` in Cell 4, or attach this repo folder as a Kaggle Dataset (then set `REPO` accordingly).

Order: env sanity (CPU) → baselines (small limits first) → SFT → GRPO smoke → eval. Keep limits tiny until memory is proven.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
!pip install -q "transformers>=5.2.0" trl peft datasets accelerate huggingface_hub jinja2
!pip uninstall -y torchao


In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(UserSecretsClient().get_secret('HF_TOKEN'))
print('HF login ok')

In [ ]:
import os, glob
# Finds repo wherever you put it: working dir, git clone, or attached dataset.
hits = glob.glob('/kaggle/working/**/data/generate_tasks.py', recursive=True) + glob.glob('/kaggle/input/**/data/generate_tasks.py', recursive=True)
import pathlib
REPO = str(pathlib.Path(hits[0]).parent.parent) if hits else '/kaggle/working/hyde'
print('REPO =', REPO)
%cd {REPO}
!ls


In [ ]:
!python data/generate_tasks.py --n-train 400 --n-dev 60 --n-test 100
!python evals/benchmark.py --tasks data/tasks/frozen_test.jsonl --limit 20  # expect 10.0 / 1.0 oracle ceiling

In [ ]:
# Baseline 1: Qwen3-0.6B smoke, 10 tasks first (fast, proves tool loop + VRAM)
!python evals/run_model_baseline.py --model Qwen/Qwen3-0.6B --tasks data/tasks/frozen_test.jsonl --out experiments/baseline_qwen06_n10.jsonl --limit 10
!python -c "import json; r=[json.loads(l) for l in open('experiments/baseline_qwen06_n10.jsonl')]; print('avg_reward', sum(x['reward'] for x in r)/len(r), 'success', sum(x['success'] for x in r)/len(r))"

In [ ]:
# Baseline 2: Qwen3.5-0.8B-Base (primary ablation). Same 10 tasks. Watch VRAM; text path only, vision frozen (unused).
!python evals/run_model_baseline.py --model Qwen/Qwen3.5-0.8B-Base --tasks data/tasks/frozen_test.jsonl --out experiments/baseline_qwen35_n10.jsonl --limit 10
!python -c "import json; r=[json.loads(l) for l in open('experiments/baseline_qwen35_n10.jsonl')]; print('avg_reward', sum(x['reward'] for x in r)/len(r), 'success', sum(x['success'] for x in r)/len(r))"

In [ ]:
# Expand baselines to 50 once 10 works (fills model matrix for router)
!python evals/run_model_baseline.py --model Qwen/Qwen3-0.6B --tasks data/tasks/frozen_test.jsonl --out experiments/baseline_qwen06_n50.jsonl --limit 50
!python evals/run_model_baseline.py --model Qwen/Qwen3.5-0.8B-Base --tasks data/tasks/frozen_test.jsonl --out experiments/baseline_qwen35_n50.jsonl --limit 50

In [ ]:
!python trajectories/collector.py --tasks data/tasks/train.jsonl --out data/tasks/sft_demos.jsonl  # expect kept ~400

In [ ]:
# LoRA SFT on Qwen3.5-0.8B (T4-safe: batch 1 x accum 8, fp16, grad-ckpt). ~15-30 min.
!CUDA_VISIBLE_DEVICES=0 python training/sft.py --model Qwen/Qwen3.5-0.8B-Base --demos data/tasks/sft_demos.jsonl --out /kaggle/working/qwen35-08b-sft --epochs 1
# re-eval SFT ckpt
!python evals/run_model_baseline.py --model /kaggle/working/qwen35-08b-sft --tasks data/tasks/frozen_test.jsonl --out experiments/sft_n50.jsonl --limit 50

In [ ]:
# GRPO smoke: 0.6B pipeline debug, 20 tasks, G=2, 256 tokens. Goal = reward moves, not SOTA.
# If OOM: lower --max-completion to 256 (already) and --group to 2.
!CUDA_VISIBLE_DEVICES=0 python training/grpo.py --model Qwen/Qwen3-0.6B --tasks data/tasks/train.jsonl --limit 20 --group 2 --max-completion 256

In [ ]:
# GRPO real: from SFT checkpoint, 200 tasks. Run only after smoke passes.
!CUDA_VISIBLE_DEVICES=0 python training/grpo.py --model /kaggle/working/qwen35-08b-sft --tasks data/tasks/train.jsonl --limit 200 --group 4 --max-completion 512
!python evals/run_model_baseline.py --model /kaggle/working/qwen-grpo --tasks data/tasks/frozen_test.jsonl --out experiments/grpo_n50.jsonl --limit 50

In [ ]:
!python -c "
import json, glob
for p in sorted(glob.glob('experiments/*.jsonl')):
    r=[json.loads(l) for l in open(p)]
    print(f\"{p}: n={len(r)} avg_reward={sum(x['reward'] for x in r)/len(r):.2f} success={sum(x['success'] for x in r)/len(r):.2f}\")
"
# Save: File → Save Version (outputs) to keep checkpoints + experiments/*.jsonl